# 서울시 약 수요예측 및 재고관리 Colab 노트북

`flu_cleaned.csv`, `hist_cleaned.csv`, `cacp_cleaned.csv`, `temp.csv` 4개 파일을 사용해서 약 수요를 예측하고, 모델별 성능 비교와 재고 기반 권장 구매량을 확인합니다.

실행 흐름:
1. CSV 4개를 Colab 데이터 폴더에 준비
2. 데이터 병합 및 시계열 파생변수 생성
3. 기준 학습모델 4종과 학습방법별 모델 비교
4. 2026년 월별/구별/약품구분별 수요 예측
5. Streamlit 대시보드에서 검증, 예측, 재고 시뮬레이션 확인


In [ ]:
!pip -q install scikit-learn joblib plotly streamlit pyngrok xgboost optuna


In [ ]:
from pathlib import Path
import warnings

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import OrdinalEncoder

warnings.filterwarnings('ignore')

DATA_DIR = Path('/content/data')
OUTPUT_DIR = Path('/content/outputs')
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DRUG_FILES = {
    'flu': 'flu_cleaned.csv',
    'hist': 'hist_cleaned.csv',
    'cacp': 'cacp_cleaned.csv',
}

FEATURE_COLUMNS = [
    'drug_type_code', 'district_code',
    'year', 'month', 'quarter', 'month_sin', 'month_cos',
    'avg_temp', 'max_temp', 'min_temp', 'rainfall', 'max_rainfall', 'avg_wind', 'max_wind',
    'temp_range', 'cold_index',
    'qty_lag1', 'qty_lag2', 'qty_lag3', 'qty_lag12', 'qty_ma3', 'qty_ma6',
]


## 1. Colab 데이터 폴더 준비

이제 파일 업로드 셀을 사용하지 않습니다. Colab 왼쪽 파일 패널에서 `/content/data` 폴더를 만들고 아래 4개 CSV 파일을 넣어둔 뒤 실행하세요.

필요한 파일명:
- `flu_cleaned.csv`
- `hist_cleaned.csv`
- `cacp_cleaned.csv`
- `temp.csv`

Google Drive를 사용할 경우에는 다음 셀의 Drive 경로 예시를 사용하면 됩니다.


In [ ]:
REQUIRED_FILES = [
    'flu_cleaned.csv',
    'hist_cleaned.csv',
    'cacp_cleaned.csv',
    'temp.csv',
]

DATA_DIR.mkdir(parents=True, exist_ok=True)
missing_files = [name for name in REQUIRED_FILES if not (DATA_DIR / name).exists()]

print(f'현재 데이터 폴더: {DATA_DIR}')
print('현재 데이터 폴더 파일:')
for path in sorted(DATA_DIR.glob('*')):
    print('-', path.name)

if missing_files:
    print()
    print('아직 없는 파일:')
    for name in missing_files:
        print('-', name)
    print()
    print('Colab 왼쪽 파일 패널에서 /content/data 폴더에 위 파일들을 넣은 뒤 이 셀을 다시 실행하세요.')
else:
    print()
    print('필요한 CSV 4개가 모두 준비되었습니다.')


In [ ]:
# Google Drive를 사용할 경우, 이 셀의 주석을 풀고 경로를 본인 Drive 위치에 맞게 바꾸세요.
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = Path('/content/drive/MyDrive/MLPro/data')
# OUTPUT_DIR = Path('/content/drive/MyDrive/MLPro/outputs')
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# print('DATA_DIR:', DATA_DIR)
# print('OUTPUT_DIR:', OUTPUT_DIR)


## 2. 데이터 로딩 및 전처리 함수

CSV 인코딩이 달라도 읽을 수 있게 처리하고, 열 이름이 깨지는 경우를 피하려고 주요 열은 위치 기준으로 정리합니다.


In [ ]:
def read_csv_with_fallback(path: Path) -> pd.DataFrame:
    for encoding in ('utf-8-sig', 'utf-8', 'cp949'):
        try:
            return pd.read_csv(path, encoding=encoding)
        except UnicodeDecodeError:
            continue
    return pd.read_csv(path)


def load_weather() -> pd.DataFrame:
    weather = read_csv_with_fallback(DATA_DIR / 'temp.csv').copy()
    # temp.csv 열 순서: 지점, 지점명, 일시, 평균기온, 최고기온, 최저기온, 월합강수량, 일최다강수량, 평균풍속, 최대풍속
    weather = weather.iloc[:, [2, 3, 4, 5, 6, 7, 8, 9]].copy()
    weather.columns = ['date', 'avg_temp', 'max_temp', 'min_temp', 'rainfall', 'max_rainfall', 'avg_wind', 'max_wind']
    weather['date'] = pd.to_datetime(weather['date'], format='%Y-%m')
    for col in weather.columns.drop('date'):
        weather[col] = pd.to_numeric(weather[col], errors='coerce')
    return weather.sort_values('date').reset_index(drop=True)


def load_demand() -> pd.DataFrame:
    frames = []
    for drug_type, file_name in DRUG_FILES.items():
        raw = read_csv_with_fallback(DATA_DIR / file_name).copy()
        # 수요 CSV 열 순서: 일시, 시도명칭, 시군구명칭, 수량, 금액
        raw = raw.iloc[:, [0, 1, 2, 3]].copy()
        raw.columns = ['date', 'province', 'district', 'qty']
        raw['date'] = pd.to_datetime(raw['date'], format='%Y-%m')
        raw['qty'] = pd.to_numeric(raw['qty'], errors='coerce')
        raw['drug_type'] = drug_type
        frames.append(raw)

    demand = pd.concat(frames, ignore_index=True)
    demand = (
        demand.groupby(['date', 'province', 'district', 'drug_type'], as_index=False)
        .agg(qty=('qty', 'sum'))
        .sort_values(['drug_type', 'district', 'date'])
        .reset_index(drop=True)
    )
    return demand


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(['drug_type', 'district', 'date']).copy()
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['quarter'] = df['date'].dt.quarter
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['temp_range'] = df['max_temp'] - df['min_temp']
    df['cold_index'] = (df['avg_temp'] < 5).astype(int)

    grouped = df.groupby(['drug_type', 'district'], sort=False)['qty']
    for lag in (1, 2, 3, 12):
        df[f'qty_lag{lag}'] = grouped.shift(lag)
    df['qty_ma3'] = grouped.transform(lambda s: s.shift(1).rolling(3).mean())
    df['qty_ma6'] = grouped.transform(lambda s: s.shift(1).rolling(6).mean())
    return df


def prepare_training_data() -> tuple[pd.DataFrame, OrdinalEncoder]:
    demand = load_demand()
    weather = load_weather()
    df = demand.merge(weather, on='date', how='left')

    encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    encoded = encoder.fit_transform(df[['drug_type', 'district']])
    df['drug_type_code'] = encoded[:, 0]
    df['district_code'] = encoded[:, 1]

    df = add_features(df)
    df = df.dropna(subset=FEATURE_COLUMNS + ['qty']).reset_index(drop=True)
    return df, encoder


## 3. 기준 학습모델 4종 학습 및 2025년 검증

2020~2024년 데이터로 `랜덤포레스트`, `XGBoost`, `라쏘회귀`, `릿지회귀`를 학습하고, 2025년 데이터를 테스트셋으로 사용해 성능을 비교합니다. 이후 기본 2026년 예측에는 2025년 평균 MAPE가 가장 낮은 모델을 사용합니다.


In [ ]:
from sklearn.linear_model import Lasso, Ridge
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, StandardScaler

try:
    from xgboost import XGBRegressor
except ImportError:
    XGBRegressor = None

BASELINE_MODEL_COL = '\ud559\uc2b5\ubaa8\ub378'


def build_baseline_model_specs():
    specs = {
        '\ub79c\ub364\ud3ec\ub808\uc2a4\ud2b8': RandomForestRegressor(
            n_estimators=500,
            max_depth=12,
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1,
        ),
        '\ub9bf\uc9c0\ud68c\uadc0': GridSearchCV(
            estimator=Pipeline([
                ('scaler', RobustScaler()),
                ('model', Ridge()),
            ]),
            param_grid={'model__alpha': np.logspace(-4, 4, 60)},
            scoring='neg_mean_absolute_error',
            cv=3,
            n_jobs=-1,
        ),
        '\ub77c\uc3d8\ud68c\uadc0': GridSearchCV(
            estimator=Pipeline([
                ('scaler', StandardScaler()),
                ('model', Lasso(max_iter=200000, random_state=42)),
            ]),
            param_grid={'model__alpha': np.logspace(-2, 7, 50)},
            scoring='neg_mean_absolute_error',
            cv=3,
            n_jobs=-1,
        ),
    }

    if XGBRegressor is not None:
        specs['XGBoost'] = XGBRegressor(
            n_estimators=500,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            objective='reg:squarederror',
            random_state=42,
            n_jobs=-1,
            verbosity=0,
        )
    else:
        print('XGBoost\uac00 \uc124\uce58\ub418\uc5b4 \uc788\uc9c0 \uc54a\uc544 XGBoost \uae30\uc900 \ubaa8\ub378\uc740 \uac74\ub108\ub701\ub2c8\ub2e4.')
    return specs


def evaluate_one_model(test_part: pd.DataFrame, model_name: str) -> pd.DataFrame:
    rows = []
    for drug_type, part in test_part.groupby('drug_type'):
        y_true = part['qty']
        y_pred = part['predicted_qty']
        rows.append({
            BASELINE_MODEL_COL: model_name,
            'drug_type': drug_type,
            'mae': mean_absolute_error(y_true, y_pred),
            'rmse': mean_squared_error(y_true, y_pred) ** 0.5,
            'r2': r2_score(y_true, y_pred),
            'mape': np.mean(np.abs((y_true - y_pred) / y_true)) * 100,
        })
    return pd.DataFrame(rows)


def train_and_evaluate_baseline_models(df: pd.DataFrame):
    train = df[df['date'] < '2025-01-01'].copy()
    test = df[df['date'] >= '2025-01-01'].copy()

    fitted_models = {}
    metrics_list = []
    prediction_list = []

    for model_name, estimator in build_baseline_model_specs().items():
        print(f'?? ?: {model_name}')
        estimator.fit(train[FEATURE_COLUMNS], train['qty'])
        fitted_models[model_name] = estimator

        pred = test[['date', 'province', 'district', 'drug_type', 'qty']].copy()
        pred['predicted_qty'] = estimator.predict(test[FEATURE_COLUMNS]).round().astype(int)
        pred[BASELINE_MODEL_COL] = model_name
        prediction_list.append(pred)
        metrics_list.append(evaluate_one_model(pred, model_name))

    metrics_all = pd.concat(metrics_list, ignore_index=True)
    predictions_all = pd.concat(prediction_list, ignore_index=True)

    mean_mape = metrics_all.groupby(BASELINE_MODEL_COL)['mape'].mean().sort_values()
    best_model_name = mean_mape.index[0]
    best_model = fitted_models[best_model_name]
    best_test_result = predictions_all[predictions_all[BASELINE_MODEL_COL] == best_model_name].copy()

    return fitted_models, best_model, best_model_name, metrics_all, predictions_all, best_test_result


df, encoder = prepare_training_data()
baseline_models, model, selected_baseline_model_name, metrics, baseline_test_results, test_result = train_and_evaluate_baseline_models(df)

print('\ud559\uc2b5 \ub370\uc774\ud130 \ud06c\uae30:', df.shape)
print('\uae30\ubcf8 2026\ub144 \uc608\uce21\uc5d0 \uc0ac\uc6a9\ud560 \ubaa8\ub378:', selected_baseline_model_name)
display(metrics.sort_values([BASELINE_MODEL_COL, 'drug_type']).round(3))


## 4. 2026년 수요 예측

2026년 기상값은 아직 없다고 가정하고, 기존 2020~2025년의 월별 평균 기상값을 사용합니다. 기본 예측은 3번 섹션에서 2025년 평균 MAPE가 가장 낮았던 기준 학습모델로 생성합니다.


In [ ]:
def build_future_weather(weather: pd.DataFrame, future_dates: pd.DatetimeIndex) -> pd.DataFrame:
    weather = weather.copy()
    weather['month'] = weather['date'].dt.month
    monthly_weather = weather.groupby('month', as_index=False).mean(numeric_only=True)
    future = pd.DataFrame({'date': future_dates})
    future['month'] = future['date'].dt.month
    future = future.merge(monthly_weather, on='month', how='left').drop(columns=['month'])
    return future


def forecast_2026(model, train_df: pd.DataFrame, encoder: OrdinalEncoder) -> pd.DataFrame:
    history = load_demand().merge(load_weather(), on='date', how='left')
    encoded = encoder.transform(history[['drug_type', 'district']])
    history['drug_type_code'] = encoded[:, 0]
    history['district_code'] = encoded[:, 1]

    future_weather = build_future_weather(load_weather(), pd.date_range('2026-01-01', '2026-12-01', freq='MS'))
    districts = train_df[['province', 'district', 'drug_type', 'drug_type_code', 'district_code']].drop_duplicates()

    predictions = []
    working = history.copy()
    for date in future_weather['date']:
        rows = districts.copy()
        rows['date'] = date
        rows = rows.merge(future_weather[future_weather['date'] == date], on='date', how='left')
        working = pd.concat([working, rows.assign(qty=np.nan)], ignore_index=True)

        featured = add_features(working)
        current = featured[featured['date'] == date].copy()
        current['predicted_qty'] = model.predict(current[FEATURE_COLUMNS]).round().astype(int)
        current['qty'] = current['predicted_qty']
        predictions.append(current[['date', 'province', 'district', 'drug_type', 'predicted_qty']])

        key_cols = ['date', 'province', 'district', 'drug_type']
        working = working.merge(current[key_cols + ['predicted_qty']], on=key_cols, how='left')
        working['qty'] = working['qty'].fillna(working['predicted_qty'])
        working = working.drop(columns=['predicted_qty'])

    forecast = pd.concat(predictions, ignore_index=True)
    forecast['date'] = forecast['date'].dt.strftime('%Y-%m')
    forecast = forecast.rename(columns={
        'date': '일시',
        'province': '시도명칭',
        'district': '시군구명칭',
        'drug_type': '약품구분',
        'predicted_qty': '예측수량',
    })
    return forecast.sort_values(['약품구분', '시군구명칭', '일시']).reset_index(drop=True)


forecast = forecast_2026(model, df, encoder)
print('2026? ?? ?? ??:', selected_baseline_model_name)
display(forecast.head(20))


## 5. 결과 저장 및 다운로드

아래 셀을 실행하면 기준 학습모델 4종의 검증 성능 CSV, 2025년 예측 비교 CSV, 선택된 기준모델의 2026년 예측 CSV, 학습된 모델 파일이 저장되고 다운로드됩니다.


In [ ]:
baseline_metrics_path = OUTPUT_DIR / 'baseline_models_2025_validation_metrics.csv'
baseline_predictions_path = OUTPUT_DIR / 'baseline_models_2025_test_predictions.csv'
baseline_forecast_path = OUTPUT_DIR / 'baseline_best_model_2026_demand_forecast.csv'
baseline_model_path = OUTPUT_DIR / 'baseline_models.joblib'

# Save legacy-compatible file names for Streamlit and later cells.
metrics_path = OUTPUT_DIR / 'random_forest_2025_validation_metrics.csv'
forecast_path = OUTPUT_DIR / 'random_forest_2026_demand_forecast.csv'
test_path = OUTPUT_DIR / 'random_forest_2025_test_predictions.csv'
model_path = OUTPUT_DIR / 'random_forest_demand_model.joblib'

metrics.to_csv(baseline_metrics_path, index=False, encoding='utf-8-sig')
baseline_test_results.to_csv(baseline_predictions_path, index=False, encoding='utf-8-sig')
forecast.to_csv(baseline_forecast_path, index=False, encoding='utf-8-sig')

metrics.to_csv(metrics_path, index=False, encoding='utf-8-sig')
test_result.to_csv(test_path, index=False, encoding='utf-8-sig')
forecast.to_csv(forecast_path, index=False, encoding='utf-8-sig')

joblib.dump({
    'baseline_models': baseline_models,
    'selected_baseline_model_name': selected_baseline_model_name,
    'model': model,
    'encoder': encoder,
    'feature_columns': FEATURE_COLUMNS,
    'drug_files': DRUG_FILES,
}, baseline_model_path)
joblib.dump({
    'model': model,
    'selected_baseline_model_name': selected_baseline_model_name,
    'encoder': encoder,
    'feature_columns': FEATURE_COLUMNS,
    'drug_files': DRUG_FILES,
}, model_path)

print('?? ??')
print(baseline_metrics_path)
print(baseline_predictions_path)
print(baseline_forecast_path)
print(baseline_model_path)

from google.colab import files
files.download(str(baseline_metrics_path))
files.download(str(baseline_forecast_path))
files.download(str(baseline_model_path))


## 6. 학습모델별 학습방법 비교

이 섹션은 학습모델을 `랜덤포레스트`, `라쏘회귀`, `릿지회귀`로 나누고, 각 학습모델에 대해 `그리드서치`, `옵튜나`, `보팅`, `배깅`, `부스팅`, `스태킹` 학습방법을 학습하고 비교합니다.

시간이 오래 걸리면 `RUN_OPTUNA = False`로 바꾸거나 `OPTUNA_TRIALS`를 줄이면 됩니다.

또한 선형모델과 비선형모델을 섞은 `혼합앙상블` 조합도 추가로 비교합니다.


In [ ]:
from sklearn.ensemble import AdaBoostRegressor, BaggingRegressor, StackingRegressor, VotingRegressor
from sklearn.linear_model import Lasso, Ridge
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.tree import DecisionTreeRegressor

try:
    from xgboost import XGBRegressor
except ImportError:
    XGBRegressor = None

try:
    import optuna
except ImportError:
    optuna = None

RUN_GRID_SEARCH = True
RUN_OPTUNA = True
OPTUNA_TRIALS = 12
MODEL_COL = '\ud559\uc2b5\ubaa8\ub378'
METHOD_COL = '\ud559\uc2b5\ubc29\ubc95'

MODEL_RF = '\ub79c\ub364\ud3ec\ub808\uc2a4\ud2b8'
MODEL_XGB = 'XGBoost'
MODEL_HYBRID = '\ud63c\ud569\uc559\uc0c1\ube14'
MODEL_LASSO = '\ub77c\uc3d8\ud68c\uadc0'
MODEL_RIDGE = '\ub9bf\uc9c0\ud68c\uadc0'

METHOD_GRID = '\uadf8\ub9ac\ub4dc\uc11c\uce58'
METHOD_OPTUNA = '\uc635\ud29c\ub098'
METHOD_VOTING = '\ubcf4\ud305'
METHOD_BAGGING = '\ubc30\uae45'
METHOD_BOOSTING = '\ubd80\uc2a4\ud305'
METHOD_STACKING = '\uc2a4\ud0dc\ud0b9'


def make_ridge(alpha=1.0):
    return Pipeline([
        ('scaler', RobustScaler()),
        ('model', Ridge(alpha=alpha)),
    ])


def make_lasso(alpha=1.0):
    return Pipeline([
        ('scaler', StandardScaler()),
        ('model', Lasso(alpha=alpha, max_iter=200000, random_state=42)),
    ])


def make_bagging(base_estimator, n_estimators=120):
    try:
        return BaggingRegressor(estimator=base_estimator, n_estimators=n_estimators, random_state=42, n_jobs=-1)
    except TypeError:
        return BaggingRegressor(base_estimator=base_estimator, n_estimators=n_estimators, random_state=42, n_jobs=-1)


def make_adaboost(base_estimator, n_estimators=120):
    try:
        return AdaBoostRegressor(estimator=base_estimator, n_estimators=n_estimators, learning_rate=0.05, random_state=42)
    except TypeError:
        return AdaBoostRegressor(base_estimator=base_estimator, n_estimators=n_estimators, learning_rate=0.05, random_state=42)


def evaluate_predictions(test_df: pd.DataFrame, model_type: str, method_name: str) -> pd.DataFrame:
    rows = []
    for drug_type, part in test_df.groupby('drug_type'):
        y_true = part['qty']
        y_pred = part['predicted_qty']
        rows.append({
            MODEL_COL: model_type,
            METHOD_COL: method_name,
            'drug_type': drug_type,
            'mae': mean_absolute_error(y_true, y_pred),
            'rmse': mean_squared_error(y_true, y_pred) ** 0.5,
            'r2': r2_score(y_true, y_pred),
            'mape': np.mean(np.abs((y_true - y_pred) / y_true)) * 100,
        })
    return pd.DataFrame(rows)


def fit_evaluate_forecast(model_type: str, method_name: str, estimator, train_df: pd.DataFrame, test_df: pd.DataFrame):
    estimator.fit(train_df[FEATURE_COLUMNS], train_df['qty'])
    pred_df = test_df[['date', 'province', 'district', 'drug_type', 'qty']].copy()
    pred_df['predicted_qty'] = estimator.predict(test_df[FEATURE_COLUMNS]).round().astype(int)
    pred_df[MODEL_COL] = model_type
    pred_df[METHOD_COL] = method_name
    metric_df = evaluate_predictions(pred_df, model_type, method_name)
    forecast_df = forecast_2026(estimator, df, encoder)
    forecast_df[MODEL_COL] = model_type
    forecast_df[METHOD_COL] = method_name
    return estimator, metric_df, pred_df, forecast_df


train_df = df[df['date'] < '2025-01-01'].copy()
test_df = df[df['date'] >= '2025-01-01'].copy()
model_specs = []

# 1. Random Forest family
if RUN_GRID_SEARCH:
    rf_grid = GridSearchCV(
        estimator=RandomForestRegressor(random_state=42, n_jobs=-1),
        param_grid={'n_estimators': [200, 500], 'max_depth': [8, 12], 'min_samples_leaf': [1, 2]},
        scoring='neg_mean_absolute_error',
        cv=3,
        n_jobs=-1,
    )
    model_specs.append((MODEL_RF, METHOD_GRID, rf_grid))

if RUN_OPTUNA and optuna is not None:
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    optuna_train = train_df[train_df['date'] < '2024-01-01'].copy()
    optuna_valid = train_df[train_df['date'] >= '2024-01-01'].copy()

    def rf_objective(trial):
        trial_model = RandomForestRegressor(
            n_estimators=trial.suggest_int('n_estimators', 200, 700, step=100),
            max_depth=trial.suggest_int('max_depth', 6, 18),
            min_samples_leaf=trial.suggest_int('min_samples_leaf', 1, 5),
            max_features=trial.suggest_categorical('max_features', ['sqrt', 0.7, 1.0]),
            random_state=42,
            n_jobs=-1,
        )
        trial_model.fit(optuna_train[FEATURE_COLUMNS], optuna_train['qty'])
        valid_pred = trial_model.predict(optuna_valid[FEATURE_COLUMNS])
        return mean_absolute_error(optuna_valid['qty'], valid_pred)

    rf_study = optuna.create_study(direction='minimize')
    rf_study.optimize(rf_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=False)
    print('RandomForest Optuna best params:', rf_study.best_params)
    model_specs.append((MODEL_RF, METHOD_OPTUNA, RandomForestRegressor(**rf_study.best_params, random_state=42, n_jobs=-1)))

rf_bagging = RandomForestRegressor(n_estimators=500, max_depth=12, min_samples_leaf=2, random_state=42, n_jobs=-1)
model_specs.append((MODEL_RF, METHOD_BAGGING, rf_bagging))

rf_voting_estimators = [
    ('rf1', RandomForestRegressor(n_estimators=250, max_depth=10, min_samples_leaf=1, random_state=41, n_jobs=-1)),
    ('rf2', RandomForestRegressor(n_estimators=350, max_depth=14, min_samples_leaf=2, random_state=42, n_jobs=-1)),
    ('ridge', make_ridge(1.0)),
]
if XGBRegressor is not None:
    rf_voting_estimators.append(('xgb', XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=4, objective='reg:squarederror', random_state=42, n_jobs=-1)))
model_specs.append((MODEL_RF, METHOD_VOTING, VotingRegressor(estimators=rf_voting_estimators, n_jobs=-1)))

if XGBRegressor is not None:
    model_specs.append((MODEL_RF, METHOD_BOOSTING, XGBRegressor(n_estimators=500, learning_rate=0.04, max_depth=4, subsample=0.9, colsample_bytree=0.9, objective='reg:squarederror', random_state=42, n_jobs=-1)))

rf_stack_estimators = [
    ('rf', RandomForestRegressor(n_estimators=300, max_depth=12, min_samples_leaf=2, random_state=42, n_jobs=-1)),
    ('ridge', make_ridge(1.0)),
]
if XGBRegressor is not None:
    rf_stack_estimators.append(('xgb', XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=4, objective='reg:squarederror', random_state=42, n_jobs=-1)))
model_specs.append((MODEL_RF, METHOD_STACKING, StackingRegressor(estimators=rf_stack_estimators, final_estimator=Ridge(alpha=1.0), cv=3, n_jobs=-1)))

# 2. XGBoost family
if XGBRegressor is not None:
    xgb_grid = GridSearchCV(
        estimator=XGBRegressor(objective='reg:squarederror', random_state=42, n_jobs=-1, verbosity=0),
        param_grid={
            'n_estimators': [200, 500],
            'learning_rate': [0.03, 0.05],
            'max_depth': [3, 4],
            'subsample': [0.7, 0.9],
            'colsample_bytree': [0.7, 0.9],
        },
        scoring='neg_mean_absolute_error',
        cv=3,
        n_jobs=-1,
    )
    model_specs.append((MODEL_XGB, METHOD_GRID, xgb_grid))

    if RUN_OPTUNA and optuna is not None:
        def xgb_objective(trial):
            params = {
                'booster': trial.suggest_categorical('booster', ['gbtree', 'dart']),
                'n_estimators': trial.suggest_int('n_estimators', 150, 500, step=50),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
                'max_depth': trial.suggest_int('max_depth', 3, 6),
                'subsample': trial.suggest_float('subsample', 0.6, 0.9),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.9),
                'reg_lambda': trial.suggest_float('reg_lambda', 5.0, 25.0),
                'objective': 'reg:squarederror',
                'random_state': 42,
                'n_jobs': -1,
                'verbosity': 0,
            }
            if params['booster'] == 'dart':
                params['rate_drop'] = trial.suggest_float('rate_drop', 0.05, 0.25)
                params['skip_drop'] = trial.suggest_float('skip_drop', 0.3, 0.7)
            trial_model = XGBRegressor(**params)
            trial_model.fit(optuna_train[FEATURE_COLUMNS], optuna_train['qty'])
            valid_pred = trial_model.predict(optuna_valid[FEATURE_COLUMNS])
            return mean_absolute_error(optuna_valid['qty'], valid_pred)
        xgb_study = optuna.create_study(direction='minimize')
        xgb_study.optimize(xgb_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=False)
        print('XGBoost Optuna best params:', xgb_study.best_params)
        best_xgb_params = dict(xgb_study.best_params)
        best_xgb_params.update({'objective': 'reg:squarederror', 'random_state': 42, 'n_jobs': -1, 'verbosity': 0})
        model_specs.append((MODEL_XGB, METHOD_OPTUNA, XGBRegressor(**best_xgb_params)))

    xgb_base = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=4, subsample=0.7, colsample_bytree=0.7, reg_lambda=15.0, objective='reg:squarederror', random_state=42, n_jobs=-1, verbosity=0)
    model_specs.append((MODEL_XGB, METHOD_BOOSTING, xgb_base))
    model_specs.append((MODEL_XGB, METHOD_BAGGING, make_bagging(XGBRegressor(n_estimators=250, learning_rate=0.05, max_depth=4, objective='reg:squarederror', random_state=42, n_jobs=-1, verbosity=0), n_estimators=60)))
    model_specs.append((MODEL_XGB, METHOD_VOTING, VotingRegressor(estimators=[
        ('xgb_a', XGBRegressor(n_estimators=250, learning_rate=0.05, max_depth=3, objective='reg:squarederror', random_state=41, n_jobs=-1, verbosity=0)),
        ('xgb_b', XGBRegressor(n_estimators=350, learning_rate=0.03, max_depth=4, objective='reg:squarederror', random_state=42, n_jobs=-1, verbosity=0)),
        ('rf', RandomForestRegressor(n_estimators=250, max_depth=12, min_samples_leaf=2, random_state=42, n_jobs=-1)),
    ], n_jobs=-1)))
    model_specs.append((MODEL_XGB, METHOD_STACKING, StackingRegressor(estimators=[
        ('xgb', XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=4, objective='reg:squarederror', random_state=42, n_jobs=-1, verbosity=0)),
        ('rf', RandomForestRegressor(n_estimators=250, max_depth=12, min_samples_leaf=2, random_state=42, n_jobs=-1)),
        ('ridge', make_ridge(1.0)),
    ], final_estimator=Ridge(alpha=1.0), cv=3, n_jobs=-1)))

# Hybrid linear + nonlinear ensembles
if XGBRegressor is not None:
    hybrid_xgb = XGBRegressor(
        n_estimators=350,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='reg:squarederror',
        random_state=42,
        n_jobs=-1,
        verbosity=0,
    )
    hybrid_rf = RandomForestRegressor(
        n_estimators=350,
        max_depth=12,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1,
    )

    model_specs.append((MODEL_HYBRID, 'Ridge+RandomForest+XGBoost', StackingRegressor(
        estimators=[
            ('ridge', make_ridge(1.0)),
            ('rf', hybrid_rf),
            ('xgb', hybrid_xgb),
        ],
        final_estimator=Ridge(alpha=1.0),
        cv=3,
        n_jobs=-1,
    )))

    model_specs.append((MODEL_HYBRID, 'Lasso+Ridge+XGBoost', StackingRegressor(
        estimators=[
            ('lasso', make_lasso(1.0)),
            ('ridge', make_ridge(1.0)),
            ('xgb', hybrid_xgb),
        ],
        final_estimator=Ridge(alpha=1.0),
        cv=3,
        n_jobs=-1,
    )))

    model_specs.append((MODEL_HYBRID, 'Ridge+Lasso+RandomForest+XGBoost', StackingRegressor(
        estimators=[
            ('ridge', make_ridge(1.0)),
            ('lasso', make_lasso(1.0)),
            ('rf', hybrid_rf),
            ('xgb', hybrid_xgb),
        ],
        final_estimator=Ridge(alpha=1.0),
        cv=3,
        n_jobs=-1,
    )))
else:
    print('XGBoost is not installed, so hybrid ensembles with XGBoost were skipped.')

# 3. Ridge family
ridge_grid = GridSearchCV(make_ridge(), {'model__alpha': np.logspace(-4, 4, 80)}, scoring='neg_mean_absolute_error', cv=3, n_jobs=-1)
model_specs.append((MODEL_RIDGE, METHOD_GRID, ridge_grid))

if RUN_OPTUNA and optuna is not None:
    def ridge_objective(trial):
        alpha = trial.suggest_float('alpha', 1e-4, 1e4, log=True)
        trial_model = make_ridge(alpha)
        trial_model.fit(optuna_train[FEATURE_COLUMNS], optuna_train['qty'])
        valid_pred = trial_model.predict(optuna_valid[FEATURE_COLUMNS])
        return mean_absolute_error(optuna_valid['qty'], valid_pred)
    ridge_study = optuna.create_study(direction='minimize')
    ridge_study.optimize(ridge_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=False)
    print('Ridge Optuna best params:', ridge_study.best_params)
    model_specs.append((MODEL_RIDGE, METHOD_OPTUNA, make_ridge(ridge_study.best_params['alpha'])))

model_specs.append((MODEL_RIDGE, METHOD_BAGGING, make_bagging(make_ridge(1.0))))
model_specs.append((MODEL_RIDGE, METHOD_VOTING, VotingRegressor(estimators=[('ridge_a', make_ridge(0.1)), ('ridge_b', make_ridge(1.0)), ('ridge_c', make_ridge(10.0))], n_jobs=-1)))
model_specs.append((MODEL_RIDGE, METHOD_BOOSTING, make_adaboost(Ridge(alpha=1.0))))
model_specs.append((MODEL_RIDGE, METHOD_STACKING, StackingRegressor(estimators=[('ridge_a', make_ridge(0.1)), ('ridge_b', make_ridge(1.0)), ('ridge_c', make_ridge(10.0))], final_estimator=Ridge(alpha=1.0), cv=3, n_jobs=-1)))

# 4. Lasso family
lasso_grid = GridSearchCV(make_lasso(), {'model__alpha': np.logspace(-2, 7, 70)}, scoring='neg_mean_absolute_error', cv=3, n_jobs=-1)
model_specs.append((MODEL_LASSO, METHOD_GRID, lasso_grid))

if RUN_OPTUNA and optuna is not None:
    def lasso_objective(trial):
        alpha = trial.suggest_float('alpha', 1e-2, 1e7, log=True)
        trial_model = make_lasso(alpha)
        trial_model.fit(optuna_train[FEATURE_COLUMNS], optuna_train['qty'])
        valid_pred = trial_model.predict(optuna_valid[FEATURE_COLUMNS])
        return mean_absolute_error(optuna_valid['qty'], valid_pred)
    lasso_study = optuna.create_study(direction='minimize')
    lasso_study.optimize(lasso_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=False)
    print('Lasso Optuna best params:', lasso_study.best_params)
    model_specs.append((MODEL_LASSO, METHOD_OPTUNA, make_lasso(lasso_study.best_params['alpha'])))

model_specs.append((MODEL_LASSO, METHOD_BAGGING, make_bagging(make_lasso(1.0))))
model_specs.append((MODEL_LASSO, METHOD_VOTING, VotingRegressor(estimators=[('lasso_a', make_lasso(0.1)), ('lasso_b', make_lasso(1.0)), ('lasso_c', make_lasso(10.0))], n_jobs=-1)))
model_specs.append((MODEL_LASSO, METHOD_BOOSTING, make_adaboost(Lasso(alpha=1.0, max_iter=200000, random_state=42))))
model_specs.append((MODEL_LASSO, METHOD_STACKING, StackingRegressor(estimators=[('lasso_a', make_lasso(0.1)), ('lasso_b', make_lasso(1.0)), ('lasso_c', make_lasso(10.0))], final_estimator=Ridge(alpha=1.0), cv=3, n_jobs=-1)))

trained_models = {}
metrics_all = []
predictions_all = []
forecasts_all = []

for model_type, method_name, estimator in model_specs:
    print(f'\nTraining {model_type} / {method_name}...')
    fitted, metric_df, pred_df, forecast_df = fit_evaluate_forecast(model_type, method_name, estimator, train_df, test_df)
    trained_models[f'{model_type}_{method_name}'] = fitted
    metrics_all.append(metric_df)
    predictions_all.append(pred_df)
    forecasts_all.append(forecast_df)
    display(metric_df.round(3))

metrics_all = pd.concat(metrics_all, ignore_index=True)
predictions_all = pd.concat(predictions_all, ignore_index=True)
forecasts_all = pd.concat(forecasts_all, ignore_index=True)

comparison_metrics_path = OUTPUT_DIR / 'model_comparison_2025_metrics.csv'
comparison_predictions_path = OUTPUT_DIR / 'model_comparison_2025_predictions.csv'
comparison_forecast_path = OUTPUT_DIR / 'model_forecast_2026_all_methods.csv'
comparison_model_path = OUTPUT_DIR / 'model_comparison_trained_models.joblib'

metrics_all.to_csv(comparison_metrics_path, index=False, encoding='utf-8-sig')
predictions_all.to_csv(comparison_predictions_path, index=False, encoding='utf-8-sig')
forecasts_all.to_csv(comparison_forecast_path, index=False, encoding='utf-8-sig')
joblib.dump({'models': trained_models, 'encoder': encoder, 'feature_columns': FEATURE_COLUMNS, 'model_column': MODEL_COL, 'method_column': METHOD_COL}, comparison_model_path)

print('\nSaved model comparison files')
print(comparison_metrics_path)
print(comparison_predictions_path)
print(comparison_forecast_path)
print(comparison_model_path)
display(metrics_all.sort_values([MODEL_COL, METHOD_COL, 'drug_type']).round(3))


## 7. 변수 중요도 확인

선택된 기준 학습모델이 트리 계열이면 feature importance를, 선형 계열이면 계수 절댓값 기준 중요도를 확인합니다.


In [ ]:
def get_feature_importance(model, feature_columns):
    fitted = model
    if hasattr(fitted, 'best_estimator_'):
        fitted = fitted.best_estimator_

    if hasattr(fitted, 'feature_importances_'):
        return pd.DataFrame({
            'feature': feature_columns,
            'importance': fitted.feature_importances_,
        }).sort_values('importance', ascending=False).reset_index(drop=True)

    if hasattr(fitted, 'named_steps'):
        final_model = fitted.named_steps.get('model')
        if final_model is not None and hasattr(final_model, 'coef_'):
            return pd.DataFrame({
                'feature': feature_columns,
                'importance': np.abs(final_model.coef_),
            }).sort_values('importance', ascending=False).reset_index(drop=True)

    if hasattr(fitted, 'coef_'):
        return pd.DataFrame({
            'feature': feature_columns,
            'importance': np.abs(fitted.coef_),
        }).sort_values('importance', ascending=False).reset_index(drop=True)

    return pd.DataFrame(columns=['feature', 'importance'])


feature_importance = get_feature_importance(model, FEATURE_COLUMNS)
print('\ubcc0\uc218 \uc911\uc694\ub3c4 \uae30\uc900 \ubaa8\ub378:', selected_baseline_model_name)
if feature_importance.empty:
    print('\uc774 \ubaa8\ub378\uc5d0\uc11c\ub294 \ubcc0\uc218 \uc911\uc694\ub3c4\ub97c \uc9c1\uc811 \ucd94\ucd9c\ud560 \uc218 \uc5c6\uc2b5\ub2c8\ub2e4.')
else:
    display(feature_importance.head(15))


## 8. 데이터 시각화

아래 셀들은 학습/예측 이후에 실행하면 됩니다. 2025년 실제값과 예측값 비교, 2026년 예측 추세, 구별 히트맵, 변수 중요도를 확인할 수 있습니다.


In [ ]:
import plotly.express as px
import plotly.graph_objects as go

# Colab에 plotly가 없을 경우 위의 패키지 설치 셀에 plotly를 추가하거나, 아래 주석을 풀고 실행하세요.
# !pip -q install plotly


In [ ]:
# 2025년 실제 수요 vs 예측 수요: 약품구분별 월 합계 비교
validation_monthly = test_result.copy()
validation_monthly['date'] = pd.to_datetime(validation_monthly['date']).dt.strftime('%Y-%m')
validation_monthly = (
    validation_monthly.groupby(['date', 'drug_type'], as_index=False)
    .agg(actual_qty=('qty', 'sum'), predicted_qty=('predicted_qty', 'sum'))
)
validation_long = validation_monthly.melt(
    id_vars=['date', 'drug_type'],
    value_vars=['actual_qty', 'predicted_qty'],
    var_name='type',
    value_name='qty',
)
validation_long['type'] = validation_long['type'].map({
    'actual_qty': '실제 수요',
    'predicted_qty': '예측 수요',
})

fig = px.line(
    validation_long,
    x='date',
    y='qty',
    color='type',
    facet_row='drug_type',
    markers=True,
    title='2025년 실제 수요 vs 랜덤포레스트 예측 수요',
    labels={'date': '일시', 'qty': '수량', 'type': '구분', 'drug_type': '약품구분'},
)
fig.update_layout(height=720, hovermode='x unified')
fig.show()


In [ ]:
# 2026년 예측 수요 추세: 약품구분별 월 합계
forecast_monthly = forecast.copy()
forecast_monthly['예측수량'] = pd.to_numeric(forecast_monthly['예측수량'], errors='coerce')
forecast_monthly = (
    forecast_monthly.groupby(['일시', '약품구분'], as_index=False)
    .agg(total_predicted_qty=('예측수량', 'sum'))
)

fig = px.line(
    forecast_monthly,
    x='일시',
    y='total_predicted_qty',
    color='약품구분',
    markers=True,
    title='2026년 약품구분별 예측 수요 추세',
    labels={'일시': '일시', 'total_predicted_qty': '예측수량', '약품구분': '약품구분'},
)
fig.update_layout(height=500, hovermode='x unified')
fig.show()


In [ ]:
# 약품구분별 2026년 구별/월별 예측 수요 히트맵
# flu, hist, cacp를 각각 따로 그립니다.
for selected_drug_type in sorted(forecast['약품구분'].dropna().unique()):
    heatmap_df = forecast[forecast['약품구분'] == selected_drug_type].copy()
    heatmap_df['예측수량'] = pd.to_numeric(heatmap_df['예측수량'], errors='coerce')
    heatmap_pivot = heatmap_df.pivot_table(
        index='시군구명칭',
        columns='일시',
        values='예측수량',
        aggfunc='sum',
    )

    fig = px.imshow(
        heatmap_pivot,
        aspect='auto',
        color_continuous_scale='YlOrRd',
        title=f'2026년 {selected_drug_type} 구별/월별 예측 수요 히트맵',
        labels={'x': '일시', 'y': '시군구명칭', 'color': '예측수량'},
    )
    fig.update_layout(height=700)
    fig.show()


In [ ]:
# 구별 연간 예측 수요 Top 10
selected_drug_type = 'flu'

top_districts = (
    forecast[forecast['약품구분'] == selected_drug_type]
    .assign(**{'예측수량': lambda x: pd.to_numeric(x['예측수량'], errors='coerce')})
    .groupby('시군구명칭', as_index=False)['예측수량']
    .sum()
    .sort_values('예측수량', ascending=False)
    .head(10)
)

fig = px.bar(
    top_districts,
    x='예측수량',
    y='시군구명칭',
    orientation='h',
    title=f'2026년 {selected_drug_type} 연간 예측 수요 Top 10 구',
    labels={'예측수량': '예측수량', '시군구명칭': '시군구명칭'},
)
fig.update_layout(height=500, yaxis={'categoryorder': 'total ascending'})
fig.show()


In [ ]:
# 변수 중요도 시각화
feature_importance = pd.DataFrame({
    'feature': FEATURE_COLUMNS,
    'importance': model.feature_importances_,
}).sort_values('importance', ascending=False).head(15)

fig = px.bar(
    feature_importance.sort_values('importance'),
    x='importance',
    y='feature',
    orientation='h',
    title='랜덤포레스트 변수 중요도 Top 15',
    labels={'importance': '중요도', 'feature': '변수'},
)
fig.update_layout(height=550)
fig.show()


## 9. Streamlit 앱으로 실행하기

아래 셀은 Colab 안에서 실행할 수 있는 Streamlit 앱 파일을 만듭니다. 먼저 위의 학습, 예측, 결과 저장 셀까지 실행해서 `/content/outputs` 안에 예측 CSV가 저장되어 있어야 합니다.


In [ ]:
streamlit_code = 'from pathlib import Path\nimport io\n\nimport pandas as pd\nimport plotly.express as px\nimport streamlit as st\n\nst.set_page_config(page_title="\\uc11c\\uc6b8\\uc2dc \\uc57d \\uc218\\uc694\\uc608\\uce21 \\ubc0f \\uc7ac\\uace0\\uad00\\ub9ac", layout="wide")\n\nOUTPUT_DIR = Path("/content/outputs")\nFORECAST_PATH = OUTPUT_DIR / "random_forest_2026_demand_forecast.csv"\nMETRICS_PATH = OUTPUT_DIR / "random_forest_2025_validation_metrics.csv"\nTEST_PATH = OUTPUT_DIR / "random_forest_2025_test_predictions.csv"\nALL_FORECAST_PATH = OUTPUT_DIR / "model_forecast_2026_all_methods.csv"\nALL_METRICS_PATH = OUTPUT_DIR / "model_comparison_2025_metrics.csv"\nALL_TEST_PATH = OUTPUT_DIR / "model_comparison_2025_predictions.csv"\n\nCOL_DATE = "\\uc77c\\uc2dc"\nCOL_DRUG = "\\uc57d\\ud488\\uad6c\\ubd84"\nCOL_DISTRICT = "\\uc2dc\\uad70\\uad6c\\uba85\\uce6d"\nCOL_PRED = "\\uc608\\uce21\\uc218\\ub7c9"\nMODEL_COL = "\\ud559\\uc2b5\\ubaa8\\ub378"\nMETHOD_COL = "\\ud559\\uc2b5\\ubc29\\ubc95"\nCOL_STOCK = "\\ud604\\uc7ac\\uc7ac\\uace0"\nCOL_START_STOCK = "\\uc6d4\\ucd08\\uc7ac\\uace0"\nCOL_BUY = "\\uad8c\\uc7a5\\uad6c\\ub9e4\\uc218\\ub7c9"\nCOL_AFTER_BUY = "\\uad6c\\ub9e4\\ud6c4\\uc7ac\\uace0"\nCOL_END_STOCK = "\\uc6d4\\ub9d0\\uc7ac\\uace0"\nCOL_BUFFER = "\\uc548\\uc804\\uc5ec\\uc720\\uc728"\nALL_LABEL = "\\uc804\\uccb4 \\ubcf4\\uae30"\nTOTAL_LABEL = "\\uc804\\uccb4\\ud569\\uacc4"\n\nst.title("\\uc11c\\uc6b8\\uc2dc \\uc57d \\uc218\\uc694\\uc608\\uce21 \\ubc0f \\uc7ac\\uace0\\uad00\\ub9ac")\nst.caption("\\uc11c\\uc6b8\\uc2dc 25\\uac1c \\uad6c\\uc758 \\uc57d\\ud488\\uad6c\\ubd84\\ubcc4 \\uc218\\uc694\\uc608\\uce21 \\uacb0\\uacfc\\uc640 \\uc7ac\\uace0 \\uae30\\ubc18 \\uad8c\\uc7a5 \\uad6c\\ub9e4\\ub7c9\\uc744 \\ud655\\uc778\\ud569\\ub2c8\\ub2e4.")\n\nforecast_source = ALL_FORECAST_PATH if ALL_FORECAST_PATH.exists() else FORECAST_PATH\nmetrics_source = ALL_METRICS_PATH if ALL_METRICS_PATH.exists() else METRICS_PATH\ntest_source = ALL_TEST_PATH if ALL_TEST_PATH.exists() else TEST_PATH\n\nif not forecast_source.exists():\n    st.error("\\uc608\\uce21 \\uacb0\\uacfc CSV\\uac00 \\uc5c6\\uc2b5\\ub2c8\\ub2e4. Colab \\ub178\\ud2b8\\ubd81\\uc5d0\\uc11c \\ud559\\uc2b5/\\uc608\\uce21/\\uc800\\uc7a5 \\uc140\\uc744 \\uba3c\\uc800 \\uc2e4\\ud589\\ud558\\uc138\\uc694.")\n    st.stop()\n\nforecast = pd.read_csv(forecast_source)\nif MODEL_COL not in forecast.columns:\n    forecast[MODEL_COL] = "\\ub79c\\ub364\\ud3ec\\ub808\\uc2a4\\ud2b8"\nif METHOD_COL not in forecast.columns:\n    forecast[METHOD_COL] = "\\ubc30\\uae45"\nforecast[COL_PRED] = pd.to_numeric(forecast[COL_PRED], errors="coerce").fillna(0)\nforecast[COL_DATE] = forecast[COL_DATE].astype(str)\n\nmodel_types = sorted(forecast[MODEL_COL].dropna().unique())\nmethod_types = sorted(forecast[METHOD_COL].dropna().unique())\ndrug_types = sorted(forecast[COL_DRUG].dropna().unique())\ndistricts = sorted(forecast[COL_DISTRICT].dropna().unique())\n\n\ndef read_csv_with_fallback(uploaded_file):\n    raw = uploaded_file.getvalue()\n    for encoding in ("utf-8-sig", "utf-8", "cp949"):\n        try:\n            return pd.read_csv(io.BytesIO(raw), encoding=encoding)\n        except UnicodeDecodeError:\n            continue\n    return pd.read_csv(io.BytesIO(raw))\n\n\ndef normalize_inventory(inventory):\n    inventory = inventory.copy()\n    inventory.columns = [str(col).strip() for col in inventory.columns]\n    rename_map = {}\n    for col in inventory.columns:\n        lower = col.lower()\n        if col == COL_DRUG or lower in ("drug_type", "drug", "type"):\n            rename_map[col] = COL_DRUG\n        elif col == COL_DISTRICT or lower in ("district", "region", "gu"):\n            rename_map[col] = COL_DISTRICT\n        elif col in (COL_STOCK, "\\uc7ac\\uace0\\uc218\\ub7c9", "\\uc7ac\\uace0") or lower in ("stock", "inventory", "current_stock", "stock_qty"):\n            rename_map[col] = COL_STOCK\n    inventory = inventory.rename(columns=rename_map)\n    required = [COL_DRUG, COL_DISTRICT, COL_STOCK]\n    missing = [col for col in required if col not in inventory.columns]\n    if missing:\n        raise ValueError("\\uc7ac\\uace0 CSV\\uc5d0 \\ud544\\uc694\\ud55c \\uc5f4\\uc774 \\uc5c6\\uc2b5\\ub2c8\\ub2e4: " + ", ".join(missing))\n    inventory = inventory[required].copy()\n    inventory[COL_DRUG] = inventory[COL_DRUG].astype(str).str.strip().replace({\'ALL_TOTAL\': TOTAL_LABEL, \'all_total\': TOTAL_LABEL, \'total\': TOTAL_LABEL})\n    inventory[COL_DISTRICT] = inventory[COL_DISTRICT].astype(str).str.strip().replace({\'ALL_TOTAL\': TOTAL_LABEL, \'all_total\': TOTAL_LABEL, \'total\': TOTAL_LABEL})\n    inventory[COL_STOCK] = pd.to_numeric(inventory[COL_STOCK], errors="coerce").fillna(0)\n    return inventory.groupby([COL_DRUG, COL_DISTRICT], as_index=False)[COL_STOCK].sum()\n\n\ndef build_inventory_matched_forecast(forecast_df, inventory_df):\n    base = forecast_df[[MODEL_COL, METHOD_COL, COL_DATE, COL_DRUG, COL_DISTRICT, COL_PRED]].copy()\n    base[COL_PRED] = pd.to_numeric(base[COL_PRED], errors="coerce").fillna(0)\n    matched_parts = []\n\n    for _, inv_row in inventory_df.iterrows():\n        drug_label = inv_row[COL_DRUG]\n        district_label = inv_row[COL_DISTRICT]\n        part = base.copy()\n        if drug_label != TOTAL_LABEL:\n            part = part[part[COL_DRUG] == drug_label]\n        if district_label != TOTAL_LABEL:\n            part = part[part[COL_DISTRICT] == district_label]\n        if part.empty:\n            continue\n\n        grouped = (\n            part.groupby([MODEL_COL, METHOD_COL, COL_DATE], as_index=False)[COL_PRED]\n            .sum()\n        )\n        grouped[COL_DRUG] = drug_label\n        grouped[COL_DISTRICT] = district_label\n        grouped[COL_STOCK] = float(inv_row[COL_STOCK])\n        matched_parts.append(grouped[[MODEL_COL, METHOD_COL, COL_DATE, COL_DRUG, COL_DISTRICT, COL_PRED, COL_STOCK]])\n\n    if not matched_parts:\n        return pd.DataFrame(columns=[MODEL_COL, METHOD_COL, COL_DATE, COL_DRUG, COL_DISTRICT, COL_PRED, COL_STOCK])\n    return pd.concat(matched_parts, ignore_index=True)\n\n\ndef simulate_inventory(forecast_df, inventory_df, buffer_rate):\n    base = build_inventory_matched_forecast(forecast_df, inventory_df)\n    if base.empty:\n        return pd.DataFrame(columns=[MODEL_COL, METHOD_COL, COL_DATE, COL_DRUG, COL_DISTRICT, COL_START_STOCK, COL_PRED, COL_BUY, COL_AFTER_BUY, COL_END_STOCK])\n    base["date_sort"] = pd.to_datetime(base[COL_DATE], format="%Y-%m", errors="coerce")\n    base = base.sort_values([MODEL_COL, METHOD_COL, COL_DRUG, COL_DISTRICT, "date_sort"])\n    rows = []\n    for (model_type, method_type, drug, district), part in base.groupby([MODEL_COL, METHOD_COL, COL_DRUG, COL_DISTRICT], sort=False):\n        stock = float(part[COL_STOCK].iloc[0])\n        for _, row in part.iterrows():\n            predicted = float(row[COL_PRED])\n            target = predicted * (1 + buffer_rate)\n            buy = max(0.0, target - stock)\n            after_buy = stock + buy\n            end_stock = max(0.0, after_buy - predicted)\n            rows.append({MODEL_COL: model_type, METHOD_COL: method_type, COL_DATE: row[COL_DATE], COL_DRUG: drug, COL_DISTRICT: district, COL_START_STOCK: round(stock), COL_PRED: round(predicted), COL_BUY: round(buy), COL_AFTER_BUY: round(after_buy), COL_END_STOCK: round(end_stock)})\n            stock = end_stock\n    return pd.DataFrame(rows)\n\n\nwith st.sidebar:\n    st.header("\\ud544\\ud130")\n    selected_model_option = st.selectbox("\\ud559\\uc2b5\\ubaa8\\ub378 \\uc120\\ud0dd", [ALL_LABEL] + model_types)\n    selected_method_option = st.selectbox("\\ud559\\uc2b5\\ubc29\\ubc95 \\uc120\\ud0dd", [ALL_LABEL] + method_types)\n    selected_heatmap_option = st.selectbox("\\ud788\\ud2b8\\ub9f5 \\uc57d\\ud488\\uad6c\\ubd84", [ALL_LABEL] + drug_types)\n    selected_drugs = st.multiselect("\\uc57d\\ud488\\uad6c\\ubd84 \\uc120\\ud0dd", drug_types, default=drug_types)\n    selected_districts = st.multiselect("\\uad6c \\uc120\\ud0dd", districts, default=districts)\n    st.divider()\n    st.header("\\uac00\\uc0c1 \\uc7ac\\uace0")\n    inventory_file = st.file_uploader("\\uc7ac\\uace0 CSV \\uc5c5\\ub85c\\ub4dc", type=["csv"])\n    buffer_rate = st.slider("\\uc608\\uce21 \\uc218\\uc694\\ubcf4\\ub2e4 \\ub354 \\uc900\\ube44\\ud560 \\ube44\\uc728", 0, 50, 10, 5) / 100\n    template = forecast[[COL_DRUG, COL_DISTRICT]].drop_duplicates().sort_values([COL_DRUG, COL_DISTRICT]).copy()\n    template[COL_STOCK] = 0\n    st.download_button("\\uc7ac\\uace0 \\ud15c\\ud50c\\ub9bf \\ub2e4\\uc6b4\\ub85c\\ub4dc", template.to_csv(index=False, encoding="utf-8-sig"), "virtual_inventory_template.csv", "text/csv")\n\nselected_models = model_types if selected_model_option == ALL_LABEL else [selected_model_option]\nselected_methods = method_types if selected_method_option == ALL_LABEL else [selected_method_option]\nheatmap_drugs = drug_types if selected_heatmap_option == ALL_LABEL else [selected_heatmap_option]\nfiltered = forecast[forecast[MODEL_COL].isin(selected_models) & forecast[METHOD_COL].isin(selected_methods) & forecast[COL_DRUG].isin(selected_drugs) & forecast[COL_DISTRICT].isin(selected_districts)].copy()\n\ntab_validation, tab_forecast, tab_inventory, tab_heatmap = st.tabs(["\\uac80\\uc99d \\ube44\\uad50", "2026 \\uc608\\uce21 \\ucd94\\uc138", "\\uc7ac\\uace0 \\uc2dc\\ubbac\\ub808\\uc774\\uc158", "\\ud788\\ud2b8\\ub9f5"])\n\nwith tab_validation:\n    if metrics_source.exists():\n        st.subheader("2025\\ub144 \\uac80\\uc99d \\uc131\\ub2a5")\n        metrics = pd.read_csv(metrics_source)\n        if MODEL_COL not in metrics.columns:\n            metrics[MODEL_COL] = "\\ub79c\\ub364\\ud3ec\\ub808\\uc2a4\\ud2b8"\n        if METHOD_COL not in metrics.columns:\n            metrics[METHOD_COL] = "\\ubc30\\uae45"\n        metrics = metrics[metrics[MODEL_COL].isin(selected_models) & metrics[METHOD_COL].isin(selected_methods)].copy()\n        if \'drug_type\' in metrics.columns:\n            metrics = metrics[metrics[\'drug_type\'].isin(selected_drugs)].copy()\n        st.dataframe(metrics, use_container_width=True)\n        if {\'mape\', MODEL_COL, METHOD_COL, \'drug_type\'}.issubset(metrics.columns):\n            metrics[\'label\'] = metrics[MODEL_COL] + \' / \' + metrics[METHOD_COL]\n            fig = px.bar(metrics, x=\'drug_type\', y=\'mape\', color=\'label\', barmode=\'group\', title=\'MAPE \\ube44\\uad50\')\n            st.plotly_chart(fig, use_container_width=True)\n    if test_source.exists():\n        test = pd.read_csv(test_source)\n        if MODEL_COL not in test.columns:\n            test[MODEL_COL] = "\\ub79c\\ub364\\ud3ec\\ub808\\uc2a4\\ud2b8"\n        if METHOD_COL not in test.columns:\n            test[METHOD_COL] = "\\ubc30\\uae45"\n        test = test[test[MODEL_COL].isin(selected_models) & test[METHOD_COL].isin(selected_methods)].copy()\n        if \'drug_type\' in test.columns:\n            test = test[test[\'drug_type\'].isin(selected_drugs)].copy()\n        if \'district\' in test.columns:\n            test = test[test[\'district\'].isin(selected_districts)].copy()\n        test[\'date\'] = pd.to_datetime(test[\'date\']).dt.strftime(\'%Y-%m\')\n        test_monthly = test.groupby([\'date\', MODEL_COL, METHOD_COL, \'drug_type\'], as_index=False).agg(actual_qty=(\'qty\', \'sum\'), predicted_qty=(\'predicted_qty\', \'sum\'))\n        test_long = test_monthly.melt(id_vars=[\'date\', MODEL_COL, METHOD_COL, \'drug_type\'], value_vars=[\'actual_qty\', \'predicted_qty\'], var_name=\'type\', value_name=\'qty\')\n        test_long[\'type\'] = test_long[\'type\'].map({\'actual_qty\':\'\\uc2e4\\uc81c \\uc218\\uc694\', \'predicted_qty\':\'\\uc608\\uce21 \\uc218\\uc694\'})\n        for model_type in selected_models:\n            for method_type in selected_methods:\n                plot_df = test_long[(test_long[MODEL_COL] == model_type) & (test_long[METHOD_COL] == method_type)]\n                if plot_df.empty:\n                    continue\n                fig = px.line(plot_df, x=\'date\', y=\'qty\', color=\'type\', facet_row=\'drug_type\', markers=True, title=f\'{model_type} / {method_type}\')\n                fig.update_layout(height=600, hovermode=\'x unified\')\n                st.plotly_chart(fig, use_container_width=True)\n\nwith tab_forecast:\n    monthly = filtered.groupby([COL_DATE, MODEL_COL, METHOD_COL, COL_DRUG], as_index=False).agg(total_predicted_qty=(COL_PRED, \'sum\'))\n    for model_type in selected_models:\n        for method_type in selected_methods:\n            method_monthly = monthly[(monthly[MODEL_COL] == model_type) & (monthly[METHOD_COL] == method_type)]\n            if method_monthly.empty:\n                continue\n            fig = px.line(method_monthly, x=COL_DATE, y=\'total_predicted_qty\', color=COL_DRUG, markers=True, title=f\'{model_type} / {method_type}\')\n            fig.update_layout(height=420, hovermode=\'x unified\')\n            st.plotly_chart(fig, use_container_width=True)\n\nwith tab_inventory:\n    if inventory_file is None:\n        st.info("\\uc0ac\\uc774\\ub4dc\\ubc14\\uc5d0\\uc11c \\uc7ac\\uace0 CSV\\ub97c \\uc5c5\\ub85c\\ub4dc\\ud558\\uba74 CSV\\uc5d0 \\uc788\\ub294 \\uc57d\\ud488/\\uad6c \\ub610\\ub294 \\uc804\\uccb4\\ud569\\uacc4 \\ub2e8\\uc704\\ub85c\\ub9cc \\uad8c\\uc7a5 \\uad6c\\ub9e4\\ub7c9\\uc774 \\uacc4\\uc0b0\\ub429\\ub2c8\\ub2e4.")\n    else:\n        inventory = normalize_inventory(read_csv_with_fallback(inventory_file))\n        simulation = simulate_inventory(filtered, inventory, buffer_rate)\n        if simulation.empty:\n            st.warning("\\uc5c5\\ub85c\\ub4dc\\ud55c \\uc7ac\\uace0 CSV\\uc640 \\ud604\\uc7ac \\ud544\\ud130\\uc5d0 \\ub9de\\ub294 \\uc608\\uce21 \\ub370\\uc774\\ud130\\uac00 \\uc5c6\\uc2b5\\ub2c8\\ub2e4. \\uc57d\\ud488\\uad6c\\ubd84/\\uc2dc\\uad70\\uad6c\\uba85\\uce6d \\uac12\\uc744 \\ud655\\uc778\\ud558\\uc138\\uc694.")\n            st.stop()\n        st.dataframe(simulation, use_container_width=True)\n        buy_summary = simulation.groupby([COL_DATE, MODEL_COL, METHOD_COL, COL_DRUG], as_index=False)[COL_BUY].sum()\n        buy_summary[\'label\'] = buy_summary[MODEL_COL] + \' / \' + buy_summary[METHOD_COL]\n        fig = px.bar(buy_summary, x=COL_DATE, y=COL_BUY, color=\'label\', facet_row=COL_DRUG, barmode=\'group\')\n        fig.update_layout(height=720)\n        st.plotly_chart(fig, use_container_width=True)\n        st.download_button("\\uc7ac\\uace0 \\uc2dc\\ubbac\\ub808\\uc774\\uc158 \\uacb0\\uacfc \\ub2e4\\uc6b4\\ub85c\\ub4dc", simulation.to_csv(index=False, encoding=\'utf-8-sig\'), \'inventory_purchase_simulation.csv\', \'text/csv\')\n\nwith tab_heatmap:\n    for model_type in selected_models:\n        for method_type in selected_methods:\n            method_df = forecast[(forecast[MODEL_COL] == model_type) & (forecast[METHOD_COL] == method_type)]\n            for selected_drug in heatmap_drugs:\n                heatmap_df = method_df[method_df[COL_DRUG] == selected_drug]\n                if heatmap_df.empty:\n                    continue\n                pivot = heatmap_df.pivot_table(index=COL_DISTRICT, columns=COL_DATE, values=COL_PRED, aggfunc=\'sum\')\n                fig = px.imshow(pivot, aspect=\'auto\', color_continuous_scale=\'YlOrRd\', title=f\'{model_type} / {method_type} / {selected_drug}\')\n                fig.update_layout(height=620)\n                st.plotly_chart(fig, use_container_width=True)\n'

app_path = Path('/content/streamlit_app.py')
app_path.write_text(streamlit_code, encoding='utf-8')
print(f'Streamlit app file created: {app_path}')


### Colab에서 Streamlit 실행하기: ngrok 사용

Colab의 보안 비밀 값에 `ngrok_key`가 저장되어 있다고 가정합니다. 아래 셀은 `ngrok_key`를 읽어서 ngrok 인증을 설정한 뒤, Streamlit 서버를 실행하고 접속 URL을 출력합니다.


In [ ]:
from google.colab import userdata
from pyngrok import ngrok
import subprocess
import time

ngrok_key = userdata.get('ngrok_key')
if not ngrok_key:
    raise ValueError('Colab 보안 비밀에 ngrok_key가 없습니다. Colab 왼쪽 열쇠 아이콘에서 ngrok_key를 추가하세요.')

ngrok.set_auth_token(ngrok_key)

# 기존 터널이 있으면 정리합니다.
ngrok.kill()

# Streamlit 서버를 백그라운드로 실행합니다.
process = subprocess.Popen([
    'streamlit', 'run', '/content/streamlit_app.py',
    '--server.port', '8501',
    '--server.address', '0.0.0.0',
    '--server.headless', 'true',
])

time.sleep(5)
public_url = ngrok.connect(8501, bind_tls=True)
print('Streamlit 접속 URL:', public_url)
